# Meeting 2026-07-16 — ED1/Cetuximab retry: 40/10/50 split + val_loss stopping

**Retry of `report/meetings/14-07-26.ipynb`.** Same question, same two
datasets (C05 ED1, AbAgym Cetuximab heavy chain), same two base models
(`vanilla_35m`, `evo_35m`). Two changes from the original run:

1. **Split fractions: 40/10/50 instead of 80/10/10.** Test sequences go from
   44/54 (ED1/Cetuximab) up to **230/279** — enough for a less noisy
   held-out Spearman. Train shrinks from 366/447 down to **180/222**.
   (`test_frac=0.5` was chosen so ED1's test set lands near ~200 rows; since
   Cetuximab's raw pool is a different size (559 vs. 456 total rows), the
   same global fraction gives it 279 test rows, not ~200 — a single
   `split.test_frac` can't hit exactly 200 for both datasets at once.)

   This is a **separate, isolated `dms_config`**
   (`conf/data/dms/ed1_cetux_40_10_50.yaml`), not an edit to
   `conf/data/dms/default.yaml`. `default.yaml`'s `split:` block is global —
   editing it in place would rebuild and change the splits for
   `ed2_m22`/`ed5_m22`/`ed811_m22` too, which the main evotuning/DPO
   experiments depend on at 80/10/10. The new config also uses distinct
   dataset keys (`ed1_m22_5050`, `cetuximab_h_5050`) and a separate
   `split.output_dir`, so it can't collide with the original 80/10/10 splits
   or with 14-07-26's runs when this notebook's plotting helpers scan
   `$TRAIN_DIR` (they dedupe finished runs by `(model, n_train, seed,
   dataset_key)`, keeping only the most-recently-modified match — reusing a
   dataset key would have silently mixed the two splits' runs together).

2. **DPO checkpoint-selection / early-stopping metric: `val_loss` instead of
   `val_spearman_avg`.** Previously (`auto`, still the default everywhere
   else): the trainer picks the "best" checkpoint and counts patience-epochs
   by whichever epoch has the highest `val_spearman_avg` — a Spearman
   correlation computed on the ~46-58 raw val *sequences* (unaffected by the
   40/10/50 change, since `val_frac` stays 0.1 either way). At that N, a
   couple of borderline points swapping rank can move the correlation by
   0.1-0.2, so "best epoch by Spearman" risks tracking noise rather than
   real improvement, and can also trigger early stopping on a spurious dip.
   `val_loss` is the DPO pair loss on the held-out val *pairs* (many pairs
   per val sequence — wt-anchor/cross/within-pos/within-neg combinations
   capped at 200 by `val_pairs_cap`), i.e. substantially more effective
   samples averaged into one number, and it's literally what training
   optimizes, so it should be smoother epoch-to-epoch. The tradeoff: it's a
   preference-satisfaction loss, not the ranking-quality metric we actually
   report on test, so it's a reasonable bet at this N but not guaranteed to
   pick the checkpoint with the best test Spearman. `training.patience=5`
   (task default) is unchanged — same early-stopping patience, just watching
   a different signal. This is implemented as a new
   `training.checkpoint_selection_metric` override (`auto` |
   `val_spearman_avg` | `val_loss`, default `auto` — every other experiment's
   behavior is unaffected) in `dpo/train.py` / `lora_dpo/train.py`.

Everything else (pairing thresholds, `model.use_context=false` for
Cetuximab, model scope) is unchanged from 14-07-26.


## Dataset 1 — C05 ED1 (single-mutant M22 DMS), 40/10/50 split

Same 456-row combined ED1 pool as 14-07-26 (see that notebook for the
provenance/pooling details — unchanged here). Only the split changed: dataset
key **`ed1_m22_5050`** in `conf/data/dms/ed1_cetux_40_10_50.yaml`, same
stratified-split code (`protein_design.dms_splitting`, `hamming_distance=0`,
10 quantile bins, seed=42) but `train_frac=0.4, val_frac=0.1, test_frac=0.5`
→ expected **180 / 46 / 230** (train/val/test).

Pairing thresholds unchanged (`strong_pos_threshold=1.0`,
`strong_neg_threshold=-5.0` — same M22 assay/scale as before).


In [ ]:
# === Replicate: C05 ED1 dataset, 40/10/50 split (build if missing, else report)
from pathlib import Path
import pandas as pd

REPO_ROOT = Path("/cluster/home/mdenegri/protein-design")
DMS_CONFIG = "conf/data/dms/ed1_cetux_40_10_50.yaml"
ED1_RAW = Path("/cluster/project/infk/krause/mdenegri/protein-design/data/raw/ED1_M22_binding_enrichment_combined.csv")
ED1_SPLIT_DIR = Path("/cluster/project/infk/krause/mdenegri/protein-design/data/dms_splits_40_10_50/ed1_m22_5050")

if not ED1_RAW.exists():
    print(f"Missing: {ED1_RAW}\nRebuild with:\n  uv run python scripts/data_prep/build_ed1_m22.py")
else:
    print(f"Found combined ED1 raw file: {ED1_RAW}")

if all((ED1_SPLIT_DIR / f"{s}.csv").exists() for s in ("train", "val", "test")):
    print(f"Found splits in {ED1_SPLIT_DIR}:")
    for s in ("train", "val", "test"):
        df = pd.read_csv(ED1_SPLIT_DIR / f"{s}.csv")
        pos_frac = (df["M22_binding_enrichment_adj"] > 0).mean()
        print(f"  {s}: {len(df):4d} rows, positive frac (M22_binding_enrichment_adj>0) = {pos_frac:.3f}")
else:
    print(
        "Missing splits. Rebuild with:\n"
        "  uv run python -c \"from protein_design.dms_splitting import ensure_dataset_splits; "
        f"ensure_dataset_splits('ed1_m22_5050', '{DMS_CONFIG}')\""
    )


## Dataset 2 — AbAgym Cetuximab (heavy-chain single mutants), 40/10/50 split

Same 559-row heavy-chain mutant set as 14-07-26 (see that notebook for the
WT/scope/sign-convention verification trail — unchanged here). Only the
split changed: dataset key **`cetuximab_h_5050`** in
`conf/data/dms/ed1_cetux_40_10_50.yaml`, `train_frac=0.4, val_frac=0.1,
test_frac=0.5` → expected **222 / 58 / 279** (train/val/test).

Pairing thresholds unchanged (`strong_pos_threshold=2.5`,
`strong_neg_threshold=-0.25`, recalibrated to C05's percentile as before).
`model.use_context=false` unchanged (full-sequence PLL, no CDR-only wrapping).


In [ ]:
# === Replicate: AbAgym Cetuximab-H dataset, 40/10/50 split (build if missing, else report)
from pathlib import Path
import pandas as pd

CETUX_RAW = Path("/cluster/project/infk/krause/mdenegri/protein-design/data/raw/AbAgym_cetuximab_h_mutants.csv")
CETUX_SPLIT_DIR = Path("/cluster/project/infk/krause/mdenegri/protein-design/data/dms_splits_40_10_50/cetuximab_h_5050")

if not CETUX_RAW.exists():
    print(f"Missing: {CETUX_RAW}\nRebuild with:\n  uv run python scripts/data_prep/build_cetuximab_h.py")
else:
    df = pd.read_csv(CETUX_RAW)
    print(f"Found {CETUX_RAW} ({len(df)} rows)")
    print(f"WT heavy chain ({df['aa'].str.len().iloc[0]} aa)")
    print(df["neg_DMS_score"].describe().round(3))

if all((CETUX_SPLIT_DIR / f"{s}.csv").exists() for s in ("train", "val", "test")):
    print(f"\nFound splits in {CETUX_SPLIT_DIR}:")
    for s in ("train", "val", "test"):
        sdf = pd.read_csv(CETUX_SPLIT_DIR / f"{s}.csv")
        pos_frac = (sdf["neg_DMS_score"] > 0).mean()
        print(f"  {s}: {len(sdf):4d} rows, positive frac (neg_DMS_score>0, i.e. enhanced-affinity) = {pos_frac:.3f}")
else:
    print(
        "\nMissing splits. Rebuild with:\n"
        "  uv run python -c \"from protein_design.dms_splitting import ensure_dataset_splits; "
        f"ensure_dataset_splits('cetuximab_h_5050', '{DMS_CONFIG}')\""
    )


## Launching the DPO low-data sweep + zero-shot baselines

N grids are re-sized to each dataset's new (40%) train-split size
(`ed1_m22_5050`: 180, `cetuximab_h_5050`: 222). 3 seeds per (model, N) for
error bars, matching the original sweep's convention. Models: `vanilla_35m`,
`evo_35m` only (per this notebook's scope). Every sweep/zero-shot call
carries `data.dms_config=conf/data/dms/ed1_cetux_40_10_50.yaml` plus the
`_5050` dataset keys, and the low-data sweep additionally carries
`training.checkpoint_selection_metric=val_loss` (see the top-of-notebook
note — early stopping/checkpoint selection now follows val_loss instead of
val_spearman_avg; `training.patience=5` is unchanged).

The zero-shot baselines reuse the **existing** `zeroshot_full` per-row score
caches from 14-07-26 (`$ANALYSIS_DIR/<model>/zeroshot_full/{ed1_m22,
cetuximab_h}.csv`) for the bootstrap-CI cell below — those are deterministic,
whole-dataset (train+val+test pooled) scores independent of which split
carved up the rows, so no GPU rescoring is needed there. The
`zeroshot_{model}_..._s0` sbatch jobs below are a separate, cheap
(`num_epochs=0`) run through the actual DPO/eval pipeline against the new
`_5050` test split, needed for the point-estimate sanity check later.

Commands below are printed, not executed — run them yourself, then re-run
the "what's missing" cell to confirm.


In [ ]:
# === Low-data sweep + zero-shot launch commands (and a "what's missing" check)
import os, re, json
from pathlib import Path
import pandas as pd
import yaml

REPO_ROOT = Path("/cluster/home/mdenegri/protein-design")
MODELS = ["vanilla_35m", "evo_35m"]
SEEDS = "0,1,2"
DMS_CONFIG = "conf/data/dms/ed1_cetux_40_10_50.yaml"

DATASETS = {
    "ed1_m22_5050": {
        "test_key": "ed1_m22_5050",
        "n_grid": "20,50,100,180",
        "extra": f" data.dms_config={DMS_CONFIG}",
    },
    "cetuximab_h_5050": {
        "test_key": "cetuximab_h_5050",
        "n_grid": "20,50,100,200,222",
        "extra": (
            f" data.dms_config={DMS_CONFIG}"
            " data.delta_based.strong_pos_threshold=2.5"
            " data.delta_based.strong_neg_threshold=-0.25"
            " model.use_context=false"
        ),
    },
}

print("# --- low-data sweeps (one launcher call per dataset) ---")
for key, spec in DATASETS.items():
    print(
        f"bash_scripts/dpo_lowdata_sweep.sh --models {','.join(MODELS)} "
        f"--n {spec['n_grid']} --seeds {SEEDS} --model-preset esm2_35m --task lora_dpo "
        f"data.dpo_dataset_key={key} data.test.dataset_key={spec['test_key']}{spec['extra']} "
        f"training.checkpoint_selection_metric=val_loss"
    )

print("\n# --- zero-shot baselines (one sbatch call per model x dataset, seed 0) ---")
for key, spec in DATASETS.items():
    ts_token = key.replace("_m22", "")
    for model in MODELS:
        print(
            f"sbatch bash_scripts/dpo_lowdata.sbatch task=lora_dpo model=esm2_35m "
            f"seed=42 training.num_epochs=0 training.batch_size=16 "
            f"data.low_data.enabled=false data.pair_split.enforce_train_controlled_sizes=false "
            f"data.low_data.val_pairs_cap=200 data.low_data.val_spearman_cap=2000 "
            f"data.low_data.test_pairs_cap=1000 training.scheduler.name=cosine "
            f"training.scheduler.interval=epoch data.dpo_dataset_key={key} "
            f"data.test.dataset_key={spec['test_key']}{spec['extra']} "
            f"run.base_name=zeroshot_{model}_{ts_token}_s0 "
            + ("model.init.source=huggingface" if model == "vanilla_35m" else
               f"model.init.source=checkpoint model.init.checkpoint=$(uv run python -c \"import yaml; "
               f"print(yaml.safe_load(open('conf/analysis/models.yaml'))['models']['{model}']['checkpoint'])\")")
        )


In [ ]:
# === What's already in $TRAIN_DIR vs. what the grid above expects ===========
# _finished() reads each candidate run's own resolved_config.yaml and
# requires data.test.dataset_key == our target key (ed1_m22_5050 /
# cetuximab_h_5050), so it can't accidentally match the 80/10/10 runs from
# 14-07-26 (those resolve to dataset_key=ed1_m22/cetuximab_h) or the
# ed2/ed5 runs from 01-07-26.
def _train_dir() -> Path:
    user = os.environ.get("USER", "unknown")
    scratch = os.environ.get("SCRATCH_DIR", f"/cluster/scratch/{user}/protein-design")
    return Path(os.environ.get("TRAIN_DIR", str(Path(scratch) / "train")))

TRAIN_DIR = _train_dir()

def _run_test_dataset_key(run_dir: Path):
    cfg_path = run_dir / "resolved_config.yaml"
    try:
        c = yaml.safe_load(cfg_path.read_text())
        return (((c or {}).get("data", {}) or {}).get("test", {}) or {}).get("dataset_key")
    except OSError:
        return None
    except Exception:
        return None

def _finished(run_name: str, target_dataset_key: str) -> bool:
    pat = re.compile(r"^" + re.escape(run_name) + r"_\d{8}_\d{6}$")
    for p in TRAIN_DIR.glob(run_name + "_*"):
        if not pat.match(p.name): continue
        try:
            if not (p / "summary.json").exists(): continue
        except OSError:
            continue
        if _run_test_dataset_key(p) == target_dataset_key:
            return True
    return False

missing = []
for key, spec in DATASETS.items():
    ts_token = key.replace("_m22", "")
    for model in MODELS:
        if not _finished(f"zeroshot_{model}_{ts_token}_s0", spec["test_key"]):
            missing.append(f"zeroshot_{model}_{ts_token}_s0  [{key}]")
        for n in spec["n_grid"].split(","):
            for s in SEEDS.split(","):
                rn = f"lowdata_{model}_n{n}_s{s}"
                if not _finished(rn, spec["test_key"]):
                    missing.append(f"{rn}  [{key}]")

print(f"TRAIN_DIR = {TRAIN_DIR}")
n_expected = sum(2 + 2 * len(s["n_grid"].split(",")) * 3 for s in DATASETS.values())
if missing:
    print(f"{len(missing)} runs missing (of {n_expected} expected):")
    for m in missing[:40]:
        print(" ", m)
    if len(missing) > 40:
        print(f"  ... and {len(missing)-40} more")
else:
    print("All expected runs present.")


## DPO low-data learning curves — ED1 and Cetuximab (40/10/50), `vanilla_35m` vs `evo_35m`

Same reader/plotting logic as 14-07-26/01-07-26: scan `$TRAIN_DIR` for
finished `lowdata_<model>_n<N>_s<seed>` and
`zeroshot_<model>_<test_set>_s<seed>` run directories, read
`test_spearman_avg` from each `summary.json`, mean+/-std across seeds per
(model, N). Zero-shot values are drawn as dotted horizontal ticks (no DPO
fine-tuning, `num_epochs=0`, evaluated on the exact same held-out test
split). `test_set` labels here are `ed1_5050`/`cetuximab_h_5050` (derived
from the `_5050` dataset keys), distinct from 14-07-26's `ed1`/`cetuximab_h`
so the two splits' runs never mix in these scans. Restricted to `MODELS =
["vanilla_35m", "evo_35m"]` for this notebook's scope.


In [ ]:
# === DPO learning curve setup (shared helpers, reads from $TRAIN_DIR) =========
%matplotlib inline
import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.transforms import blended_transform_factory, ScaledTranslation
import yaml

_REPO_ROOT = REPO_ROOT
_METRIC = "test_spearman_avg"
_MODELS_RE = "|".join(re.escape(m) for m in MODELS)
# The optional trailing (?:_.*)? tolerates run.base_name suffixes beyond
# _n<N>_s<seed>, e.g. "_splitseed101" from the external-split-seed sweep
# (bash_scripts/dpo_lowdata_sweep.sh --base-suffix). Without it, those run
# directories silently fail to match and never show up in any scan.
_BASE_RE = re.compile(rf"^lowdata_(?P<model>{_MODELS_RE})_n(?P<n>\d+)_s(?P<seed>\d+)(?:_.*)?$")
_ZEROSHOT_RE = re.compile(rf"^zeroshot_(?P<model>{_MODELS_RE})_(?P<ts>.+)_s(?P<seed>\d+)$")
_TS_RE = re.compile(r"_\d{8}_\d{6}$")

_REG_LC = yaml.safe_load((_REPO_ROOT / "conf" / "analysis" / "models.yaml").read_text()).get("models", {})
def _lc_label(key): return _REG_LC.get(key, {}).get("label") or key
def _lc_color(key, i=0):
    c = _REG_LC.get(key, {}).get("color")
    cyc = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    return c if c else cyc[i % len(cyc)]

def _ts_of(run_dir: Path) -> str:
    cfg = run_dir / "resolved_config.yaml"
    key = "unknown"
    try:
        if cfg.exists():
            c = yaml.safe_load(cfg.read_text())
            key = (((c or {}).get("data", {}) or {}).get("test", {}) or {}).get("dataset_key", key)
    except OSError: pass
    except Exception: pass
    return str(key).replace("_m22", "")

def _scan_runs_lc(train_dir=TRAIN_DIR, metric=_METRIC) -> pd.DataFrame:
    best = {}
    for p in sorted(train_dir.glob("lowdata_*_n*_s*")):
        summ = p / "summary.json"
        try:
            if not summ.exists(): continue
        except OSError:
            continue
        m = _BASE_RE.match(_TS_RE.sub("", p.name))
        if not m: continue
        ts = _ts_of(p)
        key = (m.group("model"), int(m.group("n")), int(m.group("seed")), ts)
        try:
            mt = summ.stat().st_mtime
        except OSError:
            continue
        if key in best and best[key][0] >= mt: continue
        try: val = json.loads(summ.read_text()).get(metric)
        except Exception: val = None
        best[key] = (mt, str(p), val)
    rows = [{"model": k[0], "n_train": k[1], "seed": k[2], "test_set": k[3],
             metric: v[2], "run_dir": v[1]} for k, v in best.items()]
    cols = ["model", "n_train", "seed", "test_set", metric, "run_dir"]
    if not rows: return pd.DataFrame(columns=cols)
    return pd.DataFrame(rows).sort_values(["test_set", "model", "n_train", "seed"]).reset_index(drop=True)

def _scan_zeroshot_lc(train_dir=TRAIN_DIR, metric=_METRIC) -> pd.DataFrame:
    # Deterministic zero-shot point estimates, scanned from summary.json for
    # reference/sanity-checking (e.g. against the bootstrap CI cell below).
    # Uncertainty is no longer taken from here -- see the bootstrap-CI cell.
    rows = []
    for p in sorted(train_dir.glob("zeroshot_*_s*")):
        summ = p / "summary.json"
        try:
            if not summ.exists(): continue
        except OSError:
            continue
        m = _ZEROSHOT_RE.match(_TS_RE.sub("", p.name))
        if not m: continue
        try: d = json.loads(summ.read_text())
        except Exception: d = {}
        val = d.get(metric)
        n_eval = (d.get("test_n_pos") or 0) + (d.get("test_n_neg") or 0)
        rows.append({"model": m.group("model"), "test_set": m.group("ts"),
                     "seed": int(m.group("seed")), metric: val, "n_eval": n_eval})
    cols = ["model", "test_set", "seed", metric, "n_eval"]
    if not rows: return pd.DataFrame(columns=cols)
    return pd.DataFrame(rows).sort_values(["model", "test_set"]).reset_index(drop=True)

def _plot_lc(models, test_sets, df, metric=_METRIC, min_seeds=1,
             zero_shot_df=None, title="", save_path=None, total_train=None):
    fin = df[pd.to_numeric(df[metric], errors="coerce").notna()].copy()
    fin[metric] = pd.to_numeric(fin[metric])
    avail_ts = [t for t in test_sets if t in set(fin["test_set"])]
    fig, ax = plt.subplots(figsize=(8, 5))
    used_models, used_ts = [], []
    for i, mk in enumerate(models):
        color = _lc_color(mk, i)
        for ts in avail_ts:
            sub = fin[(fin["model"] == mk) & (fin["test_set"] == ts)]
            if sub.empty: continue
            agg = (sub.groupby("n_train")[metric]
                      .agg(mean="mean", std="std", n="size").reset_index()
                      .sort_values("n_train"))
            agg = agg[agg["n"] >= min_seeds]
            if agg.empty: continue
            ax.plot(agg["n_train"], agg["mean"], marker="o", color=color)
            std = agg["std"].fillna(0.0)
            ax.fill_between(agg["n_train"], agg["mean"] - std, agg["mean"] + std,
                            color=color, alpha=0.12)
            if mk not in used_models: used_models.append(mk)
            if ts not in used_ts: used_ts.append(ts)
    has_zs = False
    if zero_shot_df is not None and not zero_shot_df.empty:
        zs = zero_shot_df[pd.to_numeric(zero_shot_df[metric], errors="coerce").notna()].copy()
        zs[metric] = pd.to_numeric(zs[metric])
        trans = blended_transform_factory(ax.transAxes, ax.transData)
        seg_w, lw_zs = 0.04, 2.5
        zs_items = []
        for mk in used_models:
            color = _lc_color(mk, models.index(mk))
            for ts in avail_ts:
                sub = zs[(zs["model"] == mk) & (zs["test_set"] == ts)]
                if sub.empty: continue
                row = sub.iloc[0]
                zs_items.append((float(row[metric]), color, row.get("ci_low"), row.get("ci_high")))
        if zs_items:
            has_zs = True
            ylo, yhi = ax.get_ylim()
            tol = 0.025 * (yhi - ylo)
            zs_items.sort(key=lambda t: t[0])
            groups = []
            for val, color, ci_lo, ci_hi in zs_items:
                if groups and abs(val - groups[-1][0][-1]) < tol:
                    groups[-1][0].append(val); groups[-1][1].append(color)
                    groups[-1][2].append(ci_lo); groups[-1][3].append(ci_hi)
                else:
                    groups.append(([val], [color], [ci_lo], [ci_hi]))
            for vals, colors, ci_los, ci_his in groups:
                items = sorted(zip(vals, colors, ci_los, ci_his), key=lambda t: t[0], reverse=True)
                k = len(items)
                for r, (val, color, ci_lo, ci_hi) in enumerate(items):
                    dy_pt = ((k - 1) / 2 - r) * lw_zs
                    t = trans + ScaledTranslation(0, dy_pt / 72.0, fig.dpi_scale_trans)
                    # 95% bootstrap-CI ribbon behind the tick (2000 resamples
                    # of the raw (score, label) pairs, see the bootstrap-CI
                    # cell above) -- a percentile CI rather than a symmetric
                    # +/-SE band, since the label distributions are skewed
                    # (Check 1 above).
                    if ci_lo is not None and ci_hi is not None and np.isfinite(ci_lo) and np.isfinite(ci_hi):
                        ax.add_patch(Rectangle(
                            (-seg_w, ci_lo), seg_w, ci_hi - ci_lo, transform=t,
                            facecolor=color, edgecolor="none", alpha=0.18,
                            clip_on=False, zorder=1,
                        ))
                        label = f"[{ci_lo:.2f}, {ci_hi:.2f}]"
                        ax.text(-seg_w - 0.09, val, label, transform=t, fontsize=6.5,
                                color="0.35", ha="right", va="center", clip_on=False)
                    ax.plot([-seg_w, 0.0], [val, val], color=color, lw=lw_zs,
                            transform=t, clip_on=False, solid_capstyle="butt", zorder=2)
    if not used_models:
        print("Nothing to plot yet (no finished runs found)."); plt.close(fig); return None
    ax.set_xscale("log")
    ax.set_xlabel("# training sequences (N)")
    ax.set_ylabel("Spearman (held-out test)")
    ax.set_title(title or "DPO low-data learning curve")
    ax.axhline(0.0, color="0.7", lw=0.8, ls=":", zorder=0)
    ax.grid(True, which="both", alpha=0.2)
    mh = [Line2D([0],[0], color=_lc_color(m, models.index(m)), lw=2, label=_lc_label(m))
          for m in used_models]
    th = []
    if has_zs:
        th.append(Line2D([0],[0], color="0.3", lw=2.5, label="← zero-shot (tick=rho, band=95% bootstrap CI)"))
    leg1 = ax.legend(handles=mh, frameon=False, fontsize=9, loc="upper left")
    ax.add_artist(leg1)
    if th:
        ax.legend(handles=th, frameon=False, fontsize=9, loc="lower right")
    if total_train:
        secax = ax.secondary_xaxis(
            "top", functions=(lambda x: 100 * x / total_train, lambda p: p * total_train / 100))
        secax.set_xlabel(f"% of {total_train}-row train split")
    fig.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight"); print("Saved:", save_path)
    return fig

_lc_runs_df = _scan_runs_lc()
_lc_zeroshot = _scan_zeroshot_lc()
print(f"DPO runs found: {len(_lc_runs_df)}")
print(f"Zero-shot evals found: {len(_lc_zeroshot)}")
print(f"Test sets seen: {sorted(_lc_runs_df['test_set'].unique()) if not _lc_runs_df.empty else []}")


In [ ]:
# === Zero-shot bootstrap CI (same method as 14-07-26) =======================
# Percentile bootstrap directly on the raw (predicted score, actual label)
# pairs from the held-out test split -- 2000 resamples with replacement,
# 2.5/97.5 percentiles of the resampled rho distribution -> 95% CI.
#
# Reuses the EXISTING $ANALYSIS_DIR/<model>/zeroshot_full/{ed1_m22,
# cetuximab_h}.csv caches from 14-07-26 (built by
# scripts/analysis/score_zeroshot_full_dataset.py): those score every row of
# the pooled train+val+test raw file, which is identical content regardless
# of how the 40/10/50 split carves it up -- so no GPU rescoring is needed
# here, just filtering against the NEW test.csv.
from scipy.stats import spearmanr

ANALYSIS_DIR = Path(os.environ.get("ANALYSIS_DIR", "/cluster/project/infk/krause/mdenegri/protein-design/analysis"))
_ZS_DATASETS = {
    "ed1_m22": {"test_set": "ed1_5050", "split_dir": ED1_SPLIT_DIR, "metric_col": "M22_binding_enrichment_adj"},
    "cetuximab_h": {"test_set": "cetuximab_h_5050", "split_dir": CETUX_SPLIT_DIR, "metric_col": "neg_DMS_score"},
}
N_BOOT = 2000
_ZS_CI_COLS = ["model", "test_set", "dataset_key", "test_spearman_avg", "ci_low", "ci_high", "n"]

def _bootstrap_spearman_ci(actual, predicted, n_boot=N_BOOT, seed=0):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    n = len(actual)
    if n < 3:
        return float("nan"), float("nan"), float("nan")
    rho = float(spearmanr(predicted, actual).statistic)
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[b] = spearmanr(predicted[idx], actual[idx]).statistic
    lo, hi = np.nanpercentile(boot, [2.5, 97.5])
    return rho, float(lo), float(hi)

def _zeroshot_full_scores(model: str, dataset_key: str) -> pd.DataFrame | None:
    path = ANALYSIS_DIR / model / "zeroshot_full" / f"{dataset_key}.csv"
    if not path.exists():
        print(f"[missing] {path}\n  rebuild: sbatch bash_scripts/extract.sbatch --what zeroshot_full --model {model} --dataset {dataset_key}")
        return None
    df = pd.read_csv(path)
    df["aa"] = df["aa"].astype(str).str.strip()
    return df

_zs_ci_rows = []
for dataset_key, spec in _ZS_DATASETS.items():
    test_aa = set(pd.read_csv(spec["split_dir"] / "test.csv")["aa"].astype(str).str.strip())
    for model in MODELS:
        full_df = _zeroshot_full_scores(model, dataset_key)
        if full_df is None:
            continue
        test_df = full_df[full_df["aa"].isin(test_aa)]
        rho, lo, hi = _bootstrap_spearman_ci(test_df[spec["metric_col"]].values, test_df["score"].values)
        _zs_ci_rows.append({
            "model": model, "test_set": spec["test_set"], "dataset_key": dataset_key,
            _METRIC: rho, "ci_low": lo, "ci_high": hi, "n": len(test_df),
        })

# Explicit `columns=` fallback: if none of the zeroshot_full artifacts exist
# yet, _zs_ci_rows is [] and pd.DataFrame([]) (no columns= given) would have
# ZERO columns -- any downstream ["ci_low"] access or merge on "model"/
# "test_set" would then raise KeyError instead of just showing "no data yet".
_lc_zeroshot_ci = pd.DataFrame(_zs_ci_rows, columns=_ZS_CI_COLS) if _zs_ci_rows else pd.DataFrame(columns=_ZS_CI_COLS)

# Sanity check: this should reproduce each zeroshot_*_s0 run's
# test_spearman_avg exactly (same checkpoint, same rows, deterministic
# cdr_pll scoring) -- confirms the full-dataset artifact + NEW test-split
# filter match the actual DPO-pipeline zero-shot evaluation on the new split.
if not _lc_zeroshot.empty and not _lc_zeroshot_ci.empty:
    _cmp = _lc_zeroshot_ci.merge(
        _lc_zeroshot[["model", "test_set", _METRIC]], on=["model", "test_set"],
        suffixes=("_bootstrap", "_original"), how="left",
    )
    _cmp["diff"] = (_cmp[f"{_METRIC}_bootstrap"] - _cmp[f"{_METRIC}_original"]).abs()
    print("Sanity check vs. zeroshot_*_s0 summary.json (should be ~0):")
    print(_cmp[["model", "test_set", f"{_METRIC}_bootstrap", f"{_METRIC}_original", "diff"]].to_string(index=False))
    print()

print(_lc_zeroshot_ci.to_string(index=False))


In [ ]:
# === DPO learning curve — C05 ED1 test (40/10/50 split) ======================
_ED1_TOTAL_TRAIN = 180  # ed1_m22_5050 split.meta.json train count
_save_ed1 = str(_REPO_ROOT / "report" / "figures" / "dpo_lowdata_curve_ed1_0716_5050.pdf")
fig_lc_ed1 = _plot_lc(
    MODELS, ["ed1_5050"], _lc_runs_df,
    zero_shot_df=_lc_zeroshot_ci,
    title="DPO low-data learning curve — C05 ED1 test (40/10/50 split)",
    save_path=_save_ed1,
    total_train=_ED1_TOTAL_TRAIN,
)
if fig_lc_ed1 is not None:
    plt.show()


In [ ]:
# === DPO learning curve — AbAgym Cetuximab (heavy chain) test (40/10/50 split)
_CETUX_TOTAL_TRAIN = 222  # cetuximab_h_5050 split.meta.json train count
_save_cetux = str(_REPO_ROOT / "report" / "figures" / "dpo_lowdata_curve_cetuximab_h_0716_5050.pdf")
fig_lc_cetux = _plot_lc(
    MODELS, ["cetuximab_h_5050"], _lc_runs_df,
    zero_shot_df=_lc_zeroshot_ci,
    title="DPO low-data learning curve — AbAgym Cetuximab (heavy chain) test (40/10/50 split)",
    save_path=_save_cetux,
    total_train=_CETUX_TOTAL_TRAIN,
)
if fig_lc_cetux is not None:
    plt.show()


## External split-seed robustness (2 seeds, on top of the 40/10/50 retry)

Same idea as 14-07-26's external-split-seed section, scaled down: instead of
5 external split seeds (101-105) x 5 internal seeds x full N-grid, this uses
**2 external split seeds (101, 102) x 5 internal seeds x full N-grid**, on
the new 40/10/50 datasets. Two axes of variation, kept distinct:

- **Internal seed** (`data.low_data.seed`): subsamples the train pool to N
  *within a fixed split* — val/test never change. 5 internal seeds here
  (matching 14-07-26's external-seed sub-experiment convention), separate
  from the 3 internal seeds used for the canonical `split.seed=42` sweep
  above.
- **External split seed** (`protein_design.dms_splitting`'s
  `<dataset>_splitseed<K>` mechanism, `K != 42`): re-instantiates the entire
  train/val/test split at the same 40/10/50 fractions, a genuinely different
  test set each time, isolated under `dms_splits_40_10_50/<dataset>_splitseed<K>/`.

Budget: 2 split seeds x (4 N-values x 2 models x 5 seeds = 40) for ED1 = 80
jobs, and 2 split seeds x (5 N-values x 2 models x 5 seeds = 50) for
Cetuximab = 100 jobs — **180 jobs total**, on top of the 58 from the
canonical sweep above (238 combined).

As before, split-seed test sets differ from each other, so learning-curve
values are **not** pooled/averaged across splits directly — only the paired
evo-vs-vanilla gap *within* the same split is aggregated across splits.


In [ ]:
# === External split-seed sweep launch commands ===============================
EXT_SPLIT_SEEDS = [101, 102]
EXT_SEEDS = "0,1,2,3,4"  # 5 internal seeds, distinct from the canonical sweep's 3

print("# --- external-split-seed sweeps (one launcher call per dataset x split seed) ---")
for key, spec in DATASETS.items():
    for split_seed in EXT_SPLIT_SEEDS:
        ext_key = f"{key}_splitseed{split_seed}"
        print(
            f"bash_scripts/dpo_lowdata_sweep.sh --models {','.join(MODELS)} "
            f"--n {spec['n_grid']} --seeds {EXT_SEEDS} --model-preset esm2_35m --task lora_dpo "
            f"--base-suffix _splitseed{split_seed} "
            f"data.dpo_dataset_key={ext_key} data.test.dataset_key={ext_key}{spec['extra']} "
            f"training.checkpoint_selection_metric=val_loss"
        )


In [ ]:
# === Scan external-split-seed runs ===========================================
_EXT_BASE_TEST_SETS = ["ed1_5050", "cetuximab_h_5050"]

_ext_runs_df = _scan_runs_lc()  # re-scan TRAIN_DIR fresh
_ext_runs_df = _ext_runs_df[_ext_runs_df["test_set"].str.contains("_splitseed", na=False)].copy()
_ext_extract = _ext_runs_df["test_set"].str.extract(r"^(?P<base_test_set>.+)_splitseed(?P<split_seed>\d+)$")
_ext_runs_df["base_test_set"] = _ext_extract["base_test_set"]
_ext_runs_df["split_seed"] = pd.to_numeric(_ext_extract["split_seed"])

_n_expected_ext = 80 + 100  # ED1 + Cetuximab, see markdown above
print(f"External-split runs found so far: {len(_ext_runs_df)} (of {_n_expected_ext} expected)")
if not _ext_runs_df.empty:
    _progress = (_ext_runs_df.groupby(["base_test_set", "split_seed"])
                 .size().rename("n_runs").reset_index())
    print(_progress.to_string(index=False))


In [ ]:
# === Small multiples plot: one panel per external split seed ================
def _plot_small_multiples(models, base_test_set, split_seeds, df, metric=_METRIC,
                           min_seeds=1, title="", save_path=None):
    fin = df[pd.to_numeric(df[metric], errors="coerce").notna()].copy()
    fin[metric] = pd.to_numeric(fin[metric])
    fig, axes = plt.subplots(1, len(split_seeds), figsize=(4.2 * len(split_seeds), 4.2), sharey=True)
    axes = np.atleast_1d(axes)
    any_data = False
    for ax, seed in zip(axes, split_seeds):
        for i, mk in enumerate(models):
            color = _lc_color(mk, i)
            sub = fin[(fin["model"] == mk) & (fin["base_test_set"] == base_test_set) & (fin["split_seed"] == seed)]
            if sub.empty:
                continue
            agg = (sub.groupby("n_train")[metric].agg(mean="mean", std="std", n="size")
                      .reset_index().sort_values("n_train"))
            agg = agg[agg["n"] >= min_seeds]
            if agg.empty:
                continue
            any_data = True
            ax.plot(agg["n_train"], agg["mean"], marker="o", color=color, label=_lc_label(mk))
            std = agg["std"].fillna(0.0)
            ax.fill_between(agg["n_train"], agg["mean"] - std, agg["mean"] + std, color=color, alpha=0.15)
        ax.set_xscale("log")
        ax.set_title(f"split seed {seed}", fontsize=10)
        ax.axhline(0.0, color="0.7", lw=0.8, ls=":", zorder=0)
        ax.grid(True, which="both", alpha=0.2)
        ax.set_xlabel("N")
    if not any_data:
        print(f"Nothing to plot yet for {base_test_set} (no finished external-split runs found)."); plt.close(fig); return None
    axes[0].set_ylabel("Spearman (held-out test)")
    handles = [Line2D([0], [0], color=_lc_color(m, i), lw=2, label=_lc_label(m)) for i, m in enumerate(models)]
    fig.legend(handles=handles, loc="upper center", ncol=len(models), frameon=False, bbox_to_anchor=(0.5, 1.08))
    fig.suptitle(title, y=1.15)
    fig.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight"); print("Saved:", save_path)
    return fig

fig_sm_ed1 = _plot_small_multiples(
    MODELS, "ed1_5050", EXT_SPLIT_SEEDS, _ext_runs_df,
    title="ED1 (40/10/50) — learning curve across 2 external split seeds",
    save_path=str(_REPO_ROOT / "report" / "figures" / "dpo_lowdata_smallmultiples_ed1_0716_5050.pdf"),
)
if fig_sm_ed1 is not None:
    plt.show()

fig_sm_cetux = _plot_small_multiples(
    MODELS, "cetuximab_h_5050", EXT_SPLIT_SEEDS, _ext_runs_df,
    title="Cetuximab (40/10/50) — learning curve across 2 external split seeds",
    save_path=str(_REPO_ROOT / "report" / "figures" / "dpo_lowdata_smallmultiples_cetuximab_h_0716_5050.pdf"),
)
if fig_sm_cetux is not None:
    plt.show()


In [ ]:
# === Paired gap computation + two-band gap(N) curve ==========================
def _compute_paired_gaps(base_test_set, split_seeds, df, metric=_METRIC,
                          model_a="evo_35m", model_b="vanilla_35m"):
    fin = df[pd.to_numeric(df[metric], errors="coerce").notna()].copy()
    fin[metric] = pd.to_numeric(fin[metric])
    rows = []
    for seed in split_seeds:
        sub = fin[(fin["base_test_set"] == base_test_set) & (fin["split_seed"] == seed)]
        a = sub[sub["model"] == model_a].set_index(["n_train", "seed"])[metric]
        b = sub[sub["model"] == model_b].set_index(["n_train", "seed"])[metric]
        common_idx = a.index.intersection(b.index)
        for (n_train, low_data_seed) in common_idx:
            rows.append({
                "split_seed": seed, "n_train": n_train, "low_data_seed": low_data_seed,
                "gap": float(a.loc[(n_train, low_data_seed)] - b.loc[(n_train, low_data_seed)]),
            })
    cols = ["split_seed", "n_train", "low_data_seed", "gap"]
    return pd.DataFrame(rows, columns=cols)

def _gap_curve_stats(gaps_df):
    per_split_cols = ["split_seed", "n_train", "split_mean", "split_std", "n_seeds"]
    outer_cols = ["n_train", "mean_gap", "outer_std", "n_splits", "pooled_inner_var", "pooled_inner_std"]
    if gaps_df.empty:
        return pd.DataFrame(columns=per_split_cols), pd.DataFrame(columns=outer_cols)
    # per (split, N): mean/std of the paired gap across the 5 internal seeds
    per_split = (gaps_df.groupby(["split_seed", "n_train"])["gap"]
                 .agg(split_mean="mean", split_std="std", n_seeds="size").reset_index())
    # across splits, per N: outer std of the split-level means, pooled inner std
    outer = (per_split.groupby("n_train")
             .agg(mean_gap=("split_mean", "mean"),
                  outer_std=("split_mean", "std"),
                  n_splits=("split_mean", "size"))
             .reset_index())
    pooled_inner_var = (per_split.groupby("n_train")["split_std"]
                         .apply(lambda s: np.nanmean(s.fillna(0.0) ** 2)))
    outer = outer.merge(pooled_inner_var.rename("pooled_inner_var"), on="n_train")
    outer["pooled_inner_std"] = np.sqrt(outer["pooled_inner_var"])
    return per_split.sort_values(["split_seed", "n_train"]), outer.sort_values("n_train")

def _plot_gap_curve(gap_stats, title="", save_path=None):
    if gap_stats.empty:
        print(f"Nothing to plot yet ({title}): no paired gap data found."); return None
    fig, ax = plt.subplots(figsize=(7, 5))
    x = gap_stats["n_train"]
    y = gap_stats["mean_gap"]
    outer = gap_stats["outer_std"].fillna(0.0)
    inner = gap_stats["pooled_inner_std"].fillna(0.0)
    # NB only 2 external splits here (vs. 5 in 14-07-26) -- the outer-std
    # band is a much less reliable estimate of split-to-split spread with
    # n_splits=2; read it as a rough sanity check, not a real error bar.
    ax.fill_between(x, y - outer, y + outer, color="C0", alpha=0.18,
                     label="±1 std across 2 external splits (outer, low n)")
    ax.fill_between(x, y - inner, y + inner, color="C1", alpha=0.30,
                     label="±1 std across 5 internal seeds, pooled (inner)")
    ax.plot(x, y, marker="o", color="0.15", label="mean gap (evo_35m − vanilla_35m)")
    ax.axhline(0.0, color="0.7", lw=0.8, ls=":")
    ax.set_xscale("log")
    ax.set_xlabel("# training sequences (N)")
    ax.set_ylabel("Spearman gap (evo_35m − vanilla_35m)")
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8, loc="best")
    ax.grid(True, which="both", alpha=0.2)
    fig.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight"); print("Saved:", save_path)
    return fig

_gaps_ed1 = _compute_paired_gaps("ed1_5050", EXT_SPLIT_SEEDS, _ext_runs_df)
_gaps_cetux = _compute_paired_gaps("cetuximab_h_5050", EXT_SPLIT_SEEDS, _ext_runs_df)
_gap_per_split_ed1, _gap_curve_ed1 = _gap_curve_stats(_gaps_ed1)
_gap_per_split_cetux, _gap_curve_cetux = _gap_curve_stats(_gaps_cetux)

fig_gap_ed1 = _plot_gap_curve(
    _gap_curve_ed1, title="Gap(N) — ED1, 40/10/50 (evo_35m − vanilla_35m)",
    save_path=str(_REPO_ROOT / "report" / "figures" / "dpo_lowdata_gapcurve_ed1_0716_5050.pdf"),
)
if fig_gap_ed1 is not None:
    plt.show()

fig_gap_cetux = _plot_gap_curve(
    _gap_curve_cetux, title="Gap(N) — Cetuximab, 40/10/50 (evo_35m − vanilla_35m)",
    save_path=str(_REPO_ROOT / "report" / "figures" / "dpo_lowdata_gapcurve_cetuximab_h_0716_5050.pdf"),
)
if fig_gap_cetux is not None:
    plt.show()
